# Mission 2 — clean 59D Model-family 1차 screen (Colab)

목적: 수렴한 MLP-300과 사전 고정한 5개 후보를 같은 internal split에서 비교합니다.
**이 Notebook은 아직 full run하지 않았습니다. 성능 결과는 없습니다.**

고정: clean 59D / 기존 call_id split / seed 42 / train-only StandardScaler / threshold 0.5.
Official Validation, 재추출, tuning, calibration, ensemble은 사용하지 않습니다.
A–D 준비 후 E–J에서 원하는 후보 셀을 직접 실행하고 K에서 비교합니다.
CPU/GPU 시간은 환경을 포함한 관측값이며 알고리즘 우열을 직접 뜻하지 않습니다.

## Checkpoint 0 — Drive mount / repo root

Colab에서는 GPU runtime을 선택하고 Drive를 mount합니다. `REPO_ROOT`를 실제 repo 경로로 직접 지정하세요.
기존 Colab Notebook에 기록된 경로는 `/content/drive/MyDrive/prelim`입니다. 현재 Drive에 이 repo와 필요한 파일이 있는지 사용자가 확인해야 합니다.
자동으로 임의 Drive 폴더를 선택하지 않습니다. 로컬에서는 cwd와 부모 경로에서 repo를 확인합니다.

`src`, 기존 manifest/metadata, clean chain, `tmp/mission2_egemaps_transfer_identity.json`을 준비하세요.
원본 source는 광희가 준 공유 폴더의 ZIP 12개입니다. 원본 WAV/JSON을 개인 Drive에 재업로드하지 않습니다.
기존 공유 폴더 바로가기 경로와 Training staging helper를 재사용합니다. 공유 폴더: https://drive.google.com/drive/folders/1cL-a0SsbT8Gxo1OSLc3eTy3Rccf2zeBo
새 source/helper도 함께 업로드해야 합니다: `mission2_model_family_screen.py`, `mission2_clean_stage4.py`, `mission2_egemaps_colab.py`, `mission2_mlp_bottleneck.py`, `mission2_numerical_embedding_mlp.py`.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

REPO_ROOT = None  # Colab: 실제 repo Path를 직접 입력. 기존 기록 예: Path('/content/drive/MyDrive/prelim')
if REPO_ROOT is None:
    roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
else:
    roots = [Path(REPO_ROOT).expanduser().resolve()]
REPO_ROOT = next((root for root in roots if (root / 'src/dcc_data/mission2_clean_stage4.py').is_file()
                  and (root / 'requirements.txt').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Checkpoint 0에서 실제 mounted repo의 REPO_ROOT를 지정하세요.')
if drive is not None and not Path('/content/drive/MyDrive').is_dir():
    raise FileNotFoundError('Google Drive mount를 먼저 완료하세요.')
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('REPO_ROOT:', REPO_ROOT)

## Checkpoint A — dependency / environment

먼저 설치 여부를 확인합니다. core package는 repository requirements를 따릅니다.
optional 권장 범위: LightGBM `>=4,<5`, XGBoost `>=2,<4`, CatBoost `>=1.2,<2`, Torch `>=2,<3`, rtdl-num-embeddings `>=0.0.12,<0.1`.
이 범위는 첫 screen용 호환 API 범위이며 성능 최적 version 주장이 아닙니다.

In [ ]:
import importlib.metadata
import importlib

packages = ['numpy', 'pandas', 'pyarrow', 'librosa', 'opensmile', 'scikit-learn',
            'lightgbm', 'xgboost', 'catboost', 'torch', 'rtdl-num-embeddings']
for package in packages:
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print('MISSING_DEPENDENCY:', package)

### Dependency setup — 필요한 경우만 직접 설치

아래 cell의 `%pip` 줄은 주석 상태이므로 Run All에서도 설치하지 않습니다.
필요한 줄만 주석을 해제해 직접 실행하세요. LightGBM 설치만으로 GPU build를 보장하지 않습니다.
새 optional package를 설치한 뒤 아래 환경 확인 cell을 다시 실행하세요.
이미 import한 package를 업그레이드했다면 kernel을 restart하고 Checkpoint 0부터 다시 실행하세요.

In [ ]:
# %pip install -r {REPO_ROOT / 'requirements.txt'}
# %pip install 'opensmile==2.6.0'  # 기존 clean loader의 간접 import dependency
# %pip install 'lightgbm>=4,<5' 'xgboost>=2,<4' 'catboost>=1.2,<2'
# %pip install 'torch>=2,<3' 'rtdl-num-embeddings>=0.0.12,<0.1'
print('설치는 자동 실행하지 않습니다. 필요한 명령만 직접 실행한 뒤 Checkpoint A를 재확인하세요.')

In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
importlib.invalidate_caches()
import pandas as pd
from IPython.display import display
from dcc_data import mission2_model_family_screen as screen

print(screen.environment_info())
requests = [('mlp_reference', 'cpu'), ('lightgbm', 'gpu'), ('lightgbm', 'cpu'),
            ('xgboost_hist', 'gpu'), ('catboost', 'gpu'),
            ('numerical_embedding_mlp', 'cuda'), ('stronger_mlp', 'cpu')]
readiness = [screen.check_candidate_environment(candidate, device) for candidate, device in requests]
display(pd.DataFrame(readiness)[['candidate', 'requested_device', 'actual_device', 'backend', 'status', 'library_version']])
for item in readiness:
    if item.get('reason'):
        print(item['candidate'], item['requested_device'], item['reason'])
# GPU check는 16개 synthetic row 또는 tiny Torch tensor만 사용합니다.
# unavailable이면 E–J의 device를 사용자가 명시적으로 변경해야 합니다.

## Checkpoint B — 공유 Training ZIP 준비 + clean 59D load

기존 공유 ZIP 12개에서 manifest의 Training WAV/JSON만 `/content`에 준비합니다.
Official Validation은 staging하지 않습니다. 기존 ZIP coverage·CRC·disk·비덮어쓰기 검증 helper를 사용합니다.
이 단계는 원본 파일의 임시 복사/해제이며 feature 재추출이나 model fit은 하지 않습니다.

기존 `tmp/mission2_egemaps_transfer_identity.json`과 모든 Training 원본의 SHA-256을 대조해야 합니다.
증거가 없거나 byte/manifest identity가 다르면 fit을 시작하지 않습니다. identity를 자동 생성하지 않습니다.
그 뒤 같은 clean chain의 artifact hash·metadata·feature 순서·row/key/call/label 검증과 기존 split/scaler를 적용합니다.

공유 전송 검증에서는 생성 당시 mtime 및 추출 package version을 historical metadata로 보존합니다.
현재 Colab의 읽기 version은 A와 environment.json에 기록합니다. 캐시나 metadata를 수정하지 않습니다.
**주의:** transfer-time 원본 동일성은 검증하지만, `source_content_proof=not_collected`인 기존 clean chain의
생성 당시 byte 증거를 소급해서 증명하는 것은 아닙니다. 이 구분을 결과 contract에도 기록합니다.

In [ ]:
# 기존 공유 폴더 바로가기 경로. 실제 mount 위치가 다르면 이 값만 수정하세요.
SHARED_ARCHIVE_ROOT = REPO_ROOT / 'data/raw_original/대학부 데이터'
RAW_ROOT = Path('/content/mission2_shared_training_raw')
TRANSFER_IDENTITY_PATH = REPO_ROOT / 'tmp/mission2_egemaps_transfer_identity.json'
ARCHIVE_PATHS = screen.shared_training_archive_paths(SHARED_ARCHIVE_ROOT)

if not RAW_ROOT.resolve().is_relative_to(Path('/content')) or RAW_ROOT.resolve().is_relative_to(Path('/content/drive')):
    raise ValueError('RAW_ROOT는 Drive 밖의 임시 /content 경로여야 합니다.')
print('Shared ZIP root:', SHARED_ARCHIVE_ROOT)
print('Transfer identity:', TRANSFER_IDENTITY_PATH, TRANSFER_IDENTITY_PATH.is_file())
for archive in ARCHIVE_PATHS:
    print(archive.name, archive.is_file())

In [ ]:
context = screen.load_shared_screen(
    REPO_ROOT, ARCHIVE_PATHS, RAW_ROOT, TRANSFER_IDENTITY_PATH,
)
print('Prepared train:', context.prepared.X_train_scaled.shape)
print('Prepared internal validation:', context.prepared.X_evaluation_scaled.shape)

## Checkpoint C — contract summary

즉시 fit 전에도 identity를 재검증합니다. key/call_id는 추적용이며 모델 X에 포함하지 않습니다.

In [ ]:
contract = screen.validate_screen_contract(context)
display({key: value for key, value in contract.items() if key not in ('identity', 'feature_names')})
print('Feature order:', contract['feature_names'])
print('Scaler samples:', context.prepared.scaler.n_samples_seen_)

## Checkpoint D — frozen configs / new run

각 모델에 정확히 하나의 설정만 있습니다. Stronger sklearn MLP는 `(64,) → (128,)`만 바뀝니다.
PLR는 공식 non-lite PeriodicEmbeddings: 24D embedding, 48 frequencies, 초기 scale 0.01,
학습형 frequencies + feature별 linear + ReLU입니다. train-derived bin은 없습니다.
MLP backbone `(256,256)`, dropout 0.1, AdamW, lr 0.001, weight decay 0.0001, batch 1024, 100 epochs를 고정합니다.
validation stopping/scheduler/checkpoint 선택은 없습니다. 마지막 epoch weights를 저장합니다.

세 tree 후보는 300 rounds, lr 0.05를 사전에 고정했습니다. tree shape·regularization은 source config에 명시합니다.
숫자는 합리적인 1차 screen 설정이며 외부 논문의 최적 결과를 재현한다는 뜻이 아닙니다.

이 cell을 다시 실행하면 새 run directory를 만듭니다. 후보를 재실행하려면 새 run을 사용하세요.

In [ ]:
configs = [screen.get_candidate_config(candidate) for candidate in screen.CANDIDATES]
display(pd.DataFrame([{key: config[key] for key in ['candidate', 'model_family', 'implementation', 'config_id', 'parameters']}
                      for config in configs]))
config_evidence = pd.DataFrame([{'candidate': config['candidate'], **item}
                                for config in configs for item in config['parameter_evidence'].values()])
display(config_evidence)
RUN_ROOT = screen.create_run(REPO_ROOT, context)
from functools import partial
run_candidate = partial(screen.run_candidate, context=context, run_root=RUN_ROOT)
print('RUN_ROOT:', RUN_ROOT)

## Checkpoint E — MLP-300 reference (CPU)

과거 score는 assertion으로 사용하지 않습니다. 기존 bottleneck Notebook output만으로 prediction을 재사용할 수 없습니다.
기본은 아래 CPU fit입니다. 완전한 screen-schema reference artifact가 있다면 대신 다음 별도 cell을 직접 실행하세요.
reference 실행과 load 중 하나만 선택합니다.

In [ ]:
mlp_reference_result = run_candidate('mlp_reference', device='cpu')
display(mlp_reference_result)

### E 대안 — strict reference artifact reuse

위 fit cell을 실행하지 않은 새 run에서만 사용하세요. `REFERENCE_DIRECTORY`를 기존 `candidates/mlp_reference` 경로로 지정합니다.
contract, config, row/key/call/label, probability, derived metric이 맞지 않으면 재사용을 거부합니다.

In [ ]:
# REFERENCE_DIRECTORY = Path('사용자가 지정한 기존 screen run/candidates/mlp_reference')
# mlp_reference_result = screen.load_reference_artifact(REFERENCE_DIRECTORY, context, RUN_ROOT)
# display(mlp_reference_result)
print('Artifact reuse는 위 CPU fit 대신 선택하는 명시적 대안입니다.')

## Checkpoint F — LightGBM

GPU backend는 build 지원을 확인합니다. 실패하면 BACKEND_UNAVAILABLE이며 자동 CPU fallback은 없습니다.
CPU를 선택하려면 아래 `device='gpu'`를 `device='cpu'`로 변경하고 실행하세요.
LightGBM native CUDA build를 준비한 경우 `device='cuda'`도 지원합니다.

In [ ]:
lightgbm_result = run_candidate('lightgbm', device='gpu')
display(lightgbm_result)

## Checkpoint G — XGBoost histogram

지원 API는 XGBoost 2.x/3.x의 `tree_method='hist', device='cuda'`입니다.
synthetic probe와 fitted booster의 실제 device를 확인해 silent fallback을 거부합니다.
GPU가 없으면 CPU를 사용자가 직접 선택할 수 있습니다.

In [ ]:
xgboost_result = run_candidate('xgboost_hist', device='gpu')
display(xgboost_result)

## Checkpoint H — CatBoost

GPU: task_type=GPU, devices=0. seed 42라도 bitwise deterministic하지 않을 수 있습니다.
allow_writing_files=False이며 cat/text feature, eval_set, validation stopping을 사용하지 않습니다.
GPU가 없으면 CPU를 사용자가 직접 선택할 수 있습니다.

In [ ]:
catboost_result = run_candidate('catboost', device='gpu')
display(catboost_result)

## Checkpoint I — Numerical Embedding MLP (CUDA)

공식 rtdl-num-embeddings package의 PLR를 사용합니다. GPU가 없으면 full fit을 시작하지 않습니다.
고정 100 epochs 전체를 internal train에서 학습하고 마지막 weights를 사용합니다.
이 후보는 architecture와 optimizer recipe가 함께 바뀌므로 capacity-only control로 해석하지 않습니다.

In [ ]:
embedding_result = run_candidate('numerical_embedding_mlp', device='cuda')
display(embedding_result)

## Checkpoint J — Stronger sklearn MLP (CPU)

hidden_layer_sizes=(128,)만 변경하며 max_iter=300을 포함한 나머지 설정은 reference와 같습니다.
sklearn MLP는 GPU를 사용하지 않습니다. n_iter_, final loss, warning도 저장합니다.

In [ ]:
stronger_result = run_candidate('stronger_mlp', device='cpu')
display(stronger_result)

## Checkpoint K — comparison / paired comparison

일치하는 contract의 성공 결과에만 reference delta를 계산합니다. reference가 없으면 delta는 비어 있습니다.
후보별 실패 상태도 표시합니다. paired 값은 같은 행의 예측 비교이며 ensemble을 구현하지 않습니다.
필요한 후보가 모두 끝난 뒤 comparison.csv를 한 번 저장하세요. 기존 파일은 덮어쓰지 않습니다.

Accuracy와 errors, FP/FN·recall trade-off, train gap을 함께 해석하세요.
후보 하나의 설정 결과는 family 최대 성능을 뜻하지 않습니다. backend가 다른 runtime을 직접 순위화하지 마세요.

In [ ]:
comparison = screen.build_comparison(RUN_ROOT, context)
display(comparison)
print('Candidate artifacts:', RUN_ROOT / 'candidates')

In [ ]:
saved_comparison = screen.save_comparison(RUN_ROOT, context)
print('Saved:', RUN_ROOT / 'comparison.csv')

## Artifact / 재실행 안내

`data/processed/mission2_model_family_screen/<run_id>/`에 run_config/environment/contract/comparison과 후보별 결과를 저장합니다.
성공 후보는 validation_predictions.parquet에 key, call, truth, P(speaker=1), prediction, confidence, correct, TN/FP/FN/TP를 저장합니다.
실패·미지원 후보는 metrics/status/reason을 저장하며 성공 점수를 만들지 않습니다.

후보 모델은 저장 후 release되며 전체 model들을 동시에 유지하지 않습니다. 재시도는 새 run directory를 사용합니다.
코드/Notebook 구현 완료와 실제 competition 성능 검증은 별개입니다.

CLI도 같은 source를 사용합니다:
`python scripts/run_mission2_model_family_screen.py --repo-root <repo> --candidate lightgbm --device cpu`
다음 후보를 같은 run에 추가하려면 출력된 run_id를 `--run-id`에 지정하세요.
`--check-environment`는 competition cache 없이 설치/backend만 확인합니다.

CLI에서 같은 공유 source를 사용하려면:
`python scripts/run_mission2_model_family_screen.py --repo-root <repo> --candidate lightgbm --device gpu --shared-archive-root <shared-ZIP-directory> --raw-root /content/mission2_shared_training_raw --transfer-identity <repo>/tmp/mission2_egemaps_transfer_identity.json`


## Stage 6 closeout — saved comparison.csv

목적: clean 59D를 고정하고 model family만 변경한 first-screen.
근거: `data/processed/mission2_model_family_screen/20261005T113752Z_8ab60cd0/comparison.csv`.
동일 조건: data / rows / labels / fixed call_id split / seed 42 / internal-train-only StandardScaler / threshold 0.5 / internal Validation. Official Validation used = False.

| Candidate | Train Accuracy | Validation Accuracy | Delta vs MLP reference | Train−Val gap | Errors |
|---|---:|---:|---:|---:|---:|
| mlp_reference | 0.8767251327 | 0.8744624517 | +0.0000000000 | 0.0022626810 | 22,332 |
| lightgbm | 0.8613653198 | 0.8597062246 | -0.0147562271 | 0.0016590952 | 24,957 |
| xgboost_hist | 0.8637083660 | 0.8590991113 | -0.0153633405 | 0.0046092547 | 25,065 |
| catboost | 0.8529067406 | 0.8535844984 | -0.0208779534 | -0.0006777578 | 26,046 |
| numerical_embedding_mlp | 0.9005449304 | 0.8749009225 | +0.0004384707 | 0.0256440079 | 22,254 |
| stronger_mlp | 0.8817766671 | 0.8760364493 | +0.0015739976 | 0.0057402178 | 22,052 |

GBDT 3종은 **이번 frozen first-screen 설정에서** MLP reference보다 낮았다. GBDT family 전체의 실패로 일반화하지 않는다.
Numerical Embedding MLP는 validation gain이 작고 Train−Val gap이 크게 증가했다. Stronger MLP가 현재 **best observed 59D candidate**이나 final model로 확정하지 않는다.
Stronger MLP의 gain도 작으므로 slice/stability 검증이 필요하다. 다음 실험에서는 clean 59D와 scaler 계약을 유지하며 TabM / FT-Transformer / Tabular ResNet을 제한적으로 screen한다.
새 실험은 `mission2_complex_tabular_screen_colab.ipynb`에서 수행한다. early stopping과 optimizer를 포함한 training recipe 차이가 있어 practical model/pipeline-level first-screen으로 해석한다.
이 closeout은 저장 CSV를 읽어 추가한 Markdown이다. 기존 셀, output, RUN_ROOT와 artifact를 재생성하거나 수정하지 않았다.
